## Azure AI Foundry Chat Completion API
This notebook demonstrates how to use the Azure AI Foundry SDK for chat completions with enhanced monitoring, conversation management, and collaborative features.

## Overview
This notebook helps you understand Azure AI Foundry's chat completion capabilities with practical examples:

- Basic chat completions using Azure AI Foundry SDK
- System vs User prompt patterns
- Multi-turn conversation management
- Temperature effects on responses
- Token usage tracking

This notebook focuses on core chat completion concepts using simple, educational examples.

#### Prerequisite
Please complete **00_Setup.ipynb** before running this notebook.

### 1. Azure Authentication and Foundry Connection

In [ ]:
# Azure Authentication using Helper Module
import os
from dotenv import load_dotenv
from azure_auth_helper import authenticate_azure

# Load environment variables
load_dotenv("./.env")

# Get tenant ID from environment variables
TENANT_ID = os.getenv('AZURE_TENANT_ID')
if not TENANT_ID:
    raise ValueError("AZURE_TENANT_ID not found in .env file. Please add it to your .env file.")

print(f"🏢 Using tenant ID: {TENANT_ID}")

# Authenticate with Azure using browser authentication (interactive)
credential = authenticate_azure(
    auth_method='browser', 
    tenant_id=TENANT_ID
)

print("✅ Azure authentication successful!")
print(f"🔑 Using browser credential for authentication")

### 2. Establish Foundry Connection and Get Models

In [ ]:
# Import additional libraries and establish Foundry connection
from azure.ai.projects import AIProjectClient

# Create Foundry project client
project = AIProjectClient(
    endpoint=os.getenv("FOUNDRY_API_ENDPOINT"),
    credential=credential,
)

# Get model deployment names from environment
gpt4o_model = os.getenv('GPT4O_DEPLOYMENT_NAME', 'gpt-4o')

# Get the OpenAI client through Foundry for enhanced features
models = project.get_openai_client(api_version="2024-10-21")

print("✅ Successfully connected to Azure AI Foundry")
print(f"� Project endpoint: {os.getenv('FOUNDRY_API_ENDPOINT')}")
print(f"🤖 Using model: {gpt4o_model}")
print(f"🔑 Using browser credential for project authentication")

### 3. Understanding System vs User Prompts

**System Prompt**: Sets the overall behavior, role, and context for the AI assistant. This message:
- Defines the AI's persona and capabilities
- Establishes rules and guidelines for responses
- Sets the tone and style of interaction
- Remains consistent throughout the conversation
- Example: "You are a helpful assistant that explains complex topics in simple terms."

**User Prompt**: Contains the specific question or task from the user. This message:
- Provides the actual request or query
- Can change with each interaction
- Contains the specific information to process
- Example: "Explain how machine learning works."

**Best Practice**: Define your system prompt clearly to establish consistent behavior, then use user prompts for specific requests.

In [ ]:
# Define system prompt separately
system_prompt = "You are a helpful AI assistant. Provide clear, accurate answers and ask for clarification if needed."

print(f"📝 System prompt configured")

### 4. Basic Chat Completion Example

In [ ]:
# Basic chat completion example
user_prompt = "Who won the high jump in 2020 Olympics?"

print(f"👤 **User:** {user_prompt}")
print()

# Make the API call using Foundry client
response = models.chat.completions.create(
    model=gpt4o_model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
)

assistant_response = response.choices[0].message.content
print(f"🤖 **Assistant:** {assistant_response}")

### 5. Multi-turn Conversation with Context

#### Understanding Stateless Models
Large language models like the GPT series are **stateless** - they don't remember previous conversations or maintain any memory between API calls. Each request is completely independent.

**How Multi-turn Conversations Work:**
1. **We maintain conversation history** - Our application stores all previous messages
2. **Send full context each time** - Every API call includes the entire conversation history
3. **Model processes everything** - The model sees the full conversation context in each request
4. **Response considers all history** - The model can reference previous messages and maintain context

**Example Flow:**
- Call 1: Send [system, user_message_1] → Get assistant_response_1
- Call 2: Send [system, user_message_1, assistant_response_1, user_message_2] → Get assistant_response_2
- Call 3: Send [system, user_message_1, assistant_response_1, user_message_2, assistant_response_2, user_message_3] → Get assistant_response_3

In [ ]:
# Multi-turn conversation - maintaining conversation history
conversation_history = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
    {"role": "assistant", "content": assistant_response}
]

# Add follow-up question
follow_up = "What was their winning height?"
conversation_history.append({"role": "user", "content": follow_up})

print(f"👤 **User:** {follow_up}")
print()

# Get response with conversation context
response = models.chat.completions.create(
    model=gpt4o_model,
    messages=conversation_history
)

follow_up_response = response.choices[0].message.content
print(f"🤖 **Assistant:** {follow_up_response}")


### 6. Temperature Effects on Responses

In [ ]:
# Define system prompt for creative tasks
creative_system_prompt = "You are a creative writing assistant. Generate engaging, imaginative content while staying helpful and appropriate."

# Demonstrate different temperature settings
creative_question = "Write a short poem about artificial intelligence"
temperatures = [0.1, 0.5, 1.8]

print("🎨 **Temperature Comparison for Creative Tasks:**")
print(f"**Question:** {creative_question}")
print()

for temp in temperatures:
    response = models.chat.completions.create(
        model=gpt4o_model,
        messages=[
            {"role": "system", "content": creative_system_prompt},
            {"role": "user", "content": creative_question}
        ],
        temperature=temp
    )
    
    print(f"**Temperature {temp}:**")
    print(f"🤖 {response.choices[0].message.content}")
    print()